# ML-09 — Validation and Research Claim Audit

**Author:** Mehak Zahra  
**Lane:** Refresh / Content Opportunity Scoring

This notebook practices constructive methodology review on two findings from *The State of AI-Driven SEO, March 2026*, then applies the same standard to my Week-5 Logistic Regression. All page and client references remain pseudonymous.

## 1. Two paper findings and constructive methodology questions

### Finding 1 — “The Anatomy of Growing Content”

The paper reports that the growing cohort averaged 3,180 words and 184 days of age, versus 2,311 words and 230 days for the declining cohort—37.6% longer and 20% younger. It explicitly presents this as a large-sample observational comparison.

**Methodology question:** How exactly were “growing” and “declining” labels produced and aligned in time with word count and age? Were comparisons re-estimated within clients or content types, or with a client-group holdout, to test whether a few client/content mixes explain the aggregate gap? The reported comparison supports a measured association; a grouped or matched analysis would clarify how broadly it transfers.

### Finding 2 — refreshed mature content

The freshness analysis reports that 365+ day pages refreshed within 30 days had 3.2× higher health (10.7 to 34.5) and 57× higher impressions (71 to 4,039) than the contrasted mature-page cohort. The paper also warns that the 361+ growth-ratio bucket is tiny and unstable.

**Methodology question:** Were recently refreshed and untouched mature pages comparable *before* refresh in prior impressions, position, page type, client, and selection reason? Because editors may preferentially refresh promising pages and health partly includes visibility inputs, an observational cohort difference supports directional prioritization but not a causal “refresh produced the lift” claim without pre/post matching, an untreated comparison, or prospective assignment.

## 2. My model under a more honest split

**Before:** a stratified random page split allows pages from the same pseudonymized client into both training and test sets. Shared client behavior can make transfer look easier.

**After:** `GroupShuffleSplit` holds out seven entire clients. The model, preprocessing, features, seed, and test fraction remain otherwise unchanged. The before/after gap is therefore evidence about validation design, not a model upgrade.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = next(p for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]]
            if (p / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists())
df = pd.read_csv(ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv')
y = df['trend_direction'].eq('down').astype(int)
SEED = 42
NUMERIC = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count',
           'impressions_90d', 'clicks_90d', 'sessions_90d', 'days_with_impressions',
           'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr',
           'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
CATEGORICAL = ['competition_level', 'content_type', 'main_intent', 'age_tier',
               'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']
FEATURES = NUMERIC + CATEGORICAL

def make_model(numeric=NUMERIC):
    prep = ColumnTransformer([
        ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)),
                              ('scale', StandardScaler())]), numeric),
        ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                                  ('onehot', OneHotEncoder(handle_unknown='ignore'))]), CATEGORICAL)])
    return Pipeline([('preprocess', prep),
                     ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED))])

def precision_at_k(y_true, score, k):
    return float(np.asarray(y_true)[np.argsort(-np.asarray(score))[:k]].mean())

X = df[FEATURES]
indices = np.arange(len(df))
random_train, random_test = train_test_split(indices, test_size=.20, random_state=SEED, stratify=y)
group_train, group_test = next(GroupShuffleSplit(n_splits=1, test_size=.20, random_state=SEED)
                               .split(X, y, groups=df['client_id']))
print(f'Loaded {len(df):,} rows; seed={SEED}; features={len(FEATURES)}')

Loaded 30,000 rows; seed=42; features=25


In [2]:
rows = []
models, score_by_split = {}, {}
for name, train_idx, test_idx in [
    ('Random page split (before)', random_train, random_test),
    ('Client-group split (after)', group_train, group_test)]:
    model = make_model().fit(X.iloc[train_idx], y.iloc[train_idx])
    score = model.predict_proba(X.iloc[test_idx])[:, 1]
    truth = y.iloc[test_idx].to_numpy()
    rows.append({'validation': name, 'test_rows': len(test_idx),
                 'base_rate': truth.mean(), 'precision_at_10': precision_at_k(truth, score, 10),
                 'precision_at_50': precision_at_k(truth, score, 50),
                 'roc_auc': roc_auc_score(truth, score),
                 'average_precision': average_precision_score(truth, score)})
    models[name], score_by_split[name] = model, score
comparison = pd.DataFrame(rows).set_index('validation').round(3)
train_clients = set(df.iloc[group_train]['client_id'])
test_clients = set(df.iloc[group_test]['client_id'])
assert train_clients.isdisjoint(test_clients)
print(comparison.to_string())
print(f'Grouped split: {len(train_clients)} train clients, {len(test_clients)} test clients, overlap=0')

                            test_rows  base_rate  precision_at_10  precision_at_50  roc_auc  average_precision
validation                                                                                                    
Random page split (before)       6000      0.542              1.0             0.82    0.694              0.705
Client-group split (after)       6163      0.511              0.8             0.74    0.580              0.577
Grouped split: 25 train clients, 7 test clients, overlap=0


The random split reports ROC AUC 0.694 and Precision@50 0.82. Under unseen-client validation these fall to 0.580 and 0.74. The 0.114 AUC drop shows that random page validation materially overstated transfer. I keep the grouped result because the intended decision-support tool may face client patterns absent from training.

## 3. Leakage audit

The target is derived from `trend_direction`, which itself comes from `trend_pct` and the last/previous-30-day impression windows. Those fields and sibling click/session windows are blocked. Pseudonymous IDs are context only; existing product flags/scores are blocked decision-derived features.

One residual timing issue remains: retained trailing-90-day aggregates overlap the two 30-day periods defining the proxy. They do not directly reveal the label, but they prevent a clean future-prediction claim. The valid framing is **contemporaneous triage**.

In [3]:
FORBIDDEN = ['trend_direction', 'trend_pct', 'is_declining_label',
             'impressions_last_30d', 'impressions_prev_30d',
             'clicks_last_30d', 'clicks_prev_30d',
             'sessions_last_30d', 'sessions_prev_30d']
audit = pd.DataFrame([
 ['IDs', 'content_id, client_id', 'Context only', 'Join, identity, grouped split; never learned.'],
 ['Label', 'trend_direction', 'Blocked', 'The binary proxy is derived from this field.'],
 ['Label source', 'trend_pct', 'Blocked', 'Directly computes trend_direction.'],
 ['Sibling windows', 'last/prev 30-day measures', 'Blocked', 'They construct the trend proxy.'],
 ['Product decisions', 'existing flags/scores', 'Blocked', 'Would learn the old rule.'],
 ['90-day aggregates', 'traffic, position, rates', 'Warning', 'Overlap means triage, not forecasting.']],
 columns=['class', 'fields', 'status', 'reason'])
assert set(FEATURES).isdisjoint(FORBIDDEN)
print(audit.to_string(index=False))

                    class                    fields                status                                                                                       reason
                      IDs     content_id, client_id          Context only                                            Join, identity, and grouped split; never learned.
                    Label           trend_direction               Blocked                                                 The binary proxy is derived from this field.
             Label source                 trend_pct               Blocked            This directly computes trend_direction; deliberate test reaches near-perfect AUC.
          Sibling windows last/prev 30-day measures               Blocked                                              They construct the contemporaneous trend proxy.
        Product decisions     existing flags/scores               Blocked                     Would reproduce an existing rule rather than learn independent evidence

### Deliberate leak test

I add `trend_pct` once on purpose. It is the direct source of the target and therefore unknowable as an independent feature. If the test harness is working, grouped performance should jump toward perfect; I then delete it and retain the honest result.

In [4]:
honest_score = score_by_split['Client-group split (after)']
X_leaky = df[FEATURES + ['trend_pct']]
leaky = make_model(NUMERIC + ['trend_pct']).fit(X_leaky.iloc[group_train], y.iloc[group_train])
leaky_score = leaky.predict_proba(X_leaky.iloc[group_test])[:, 1]
truth = y.iloc[group_test].to_numpy()
leakage_comparison = pd.DataFrame([
 {'feature_set': 'honest features', 'roc_auc': roc_auc_score(truth, honest_score),
  'precision_at_50': precision_at_k(truth, honest_score, 50)},
 {'feature_set': 'with trend_pct (deliberate leak)', 'roc_auc': roc_auc_score(truth, leaky_score),
  'precision_at_50': precision_at_k(truth, leaky_score, 50)}]).set_index('feature_set').round(3)
print(leakage_comparison.to_string())
del X_leaky, leaky, leaky_score
assert 'trend_pct' not in FEATURES
print('Leak deleted; honest grouped metrics retained.')

                                  roc_auc  precision_at_50
feature_set                                               
honest features                     0.580             0.74
with trend_pct (deliberate leak)    0.999             1.00
Leak deleted; honest grouped metrics retained.


### Real failure examples

These are three confident false positives and three confident false negatives from the grouped test. They expose missing client/query context and sparse-history behavior; they do not identify any private page.

In [5]:
score = score_by_split['Client-group split (after)']
pred = (score >= .5).astype(int)
errors = df.iloc[group_test][['content_id', 'impressions_90d', 'days_with_impressions',
                              'avg_position', 'ctr', 'content_age_days']].copy()
errors['actual'], errors['predicted'], errors['probability'] = truth, pred, score
fp = errors[(errors.actual == 0) & (errors.predicted == 1)].nlargest(3, 'probability')
fn = errors[(errors.actual == 1) & (errors.predicted == 0)].nsmallest(3, 'probability')
error_cases = pd.concat([fp.assign(error_type='false_positive'), fn.assign(error_type='false_negative')])
error_cases['why_difficult'] = np.where(error_cases.error_type.eq('false_positive'),
 'Historical activity resembles risk, but the observed proxy stayed non-declining.',
 'Sparse or irregular history hides an observed decline from the model.')
error_view = error_cases[['error_type', 'content_id', 'probability', 'impressions_90d',
                          'days_with_impressions', 'avg_position', 'ctr', 'why_difficult']].copy()
error_view['probability'] = error_view['probability'].round(3)
print(error_view.to_string(index=False))

    error_type           content_id  probability  impressions_90d  days_with_impressions  avg_position  ctr                                                                    why_difficult
false_positive content_374e795aab68        0.920              235                     64          31.0 0.85 Historical activity resembles risk, but the observed proxy stayed non-declining.
false_positive content_7be5f150dc65        0.900              290                     53           5.9 0.00 Historical activity resembles risk, but the observed proxy stayed non-declining.
false_positive content_41baf0722ad9        0.884             3115                     88          12.8 0.00 Historical activity resembles risk, but the observed proxy stayed non-declining.
false_negative content_e18144cbd19d        0.065                3                      3           2.0 0.00            Sparse or irregular history hides an observed decline from the model.
false_negative content_917fc1b11fe1        0.066       

## 4. Claim rewrite

**Too strong:** “The model predicts which pages will decline and should be refreshed.”

**Evidence-aligned:** “On a held-out set of seven unseen client groups, Logistic Regression ranked the supplied contemporaneous decline proxy at Precision@50 of 0.74. The score can support human review of measurable pages; it does not forecast future Google performance or show that refreshing a recommended page will cause recovery.”

**Paper-style recommendation rewrite:** Instead of “refreshing mature pages produces the lift,” use: “In the observed portfolio cohort, recently refreshed mature pages had higher measured health and impressions than the comparison cohort. A matched or prospective follow-up is needed to estimate refresh impact.”

In [6]:
metrics = {
 'assignment': 'ML-09', 'author': 'Mehak Zahra', 'seed': SEED,
 'before_after': comparison.reset_index().to_dict(orient='records'),
 'leakage_test': leakage_comparison.reset_index().to_dict(orient='records'),
 'client_overlap_after': len(train_clients & test_clients),
 'honest_interpretation': 'contemporaneous decision-support triage, not future prediction',
 'sklearn_version': sklearn.__version__}
out = ROOT / 'work' / 'outputs'; out.mkdir(parents=True, exist_ok=True)
(out / 'validation_audit_metrics.json').write_text(json.dumps(metrics, indent=2) + '\n')
assert len(train_clients & test_clients) == 0
assert set(FEATURES).isdisjoint(FORBIDDEN)
print('Validation checks passed; wrote work/outputs/validation_audit_metrics.json')

Validation checks passed; wrote work/outputs/validation_audit_metrics.json


## 5. Self-check

- [x] Two named paper findings have respectful, concrete methodology questions.
- [x] The same model is shown before and after client-group validation.
- [x] Base rate and ranking metrics appear beside both splits.
- [x] Leakage taxonomy, timeline warning, deliberate leak test, and removal are visible.
- [x] Six real pseudonymous failure cases are inspected.
- [x] Overstated model and refresh claims are rewritten using observed/measured/directional/decision-support language.
- [x] All code cells are executed and outputs are visible.